# Notebook 01: Análise Exploratória, Geoespacial e Correlação Climática

## Objetivo
Extrair dados do 1746 (via BigQuery), integrar com APIs meteorológicas e feriados, analisar correlações e plotar a distribuição espacial dos chamados na cidade do Rio de Janeiro.

### Perguntas de Negócio Sênior:
1. **Como o clima afeta os chamados?** Esperamos ver picos de 'Alagamentos' e 'Poda de Árvore' em dias com alta precipitação.
2. **Existe sazonalidade espacial?** Queremos verificar se certas Áreas de Planejamento (APs) sofrem mais em condições extremas.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
from data_fetcher import DataFetcher
from features import DateFeaturesExtractor

import warnings
warnings.filterwarnings('ignore')

## 1. Extração de Dados do 1746 (BigQuery)

In [ ]:
# Instancia o extrator
fetcher = DataFetcher()

# Extração com limites para EDA e garantindo D>=2023
df_1746 = fetcher.get_1746_data(start_date='2023-01-01', limit=50000)

df_1746.head()

## 2. Integração com APIs Externa (Clima e Feriados)

In [ ]:
# Busca assíncrona/síncrona de clima (simulação usando wrapper)
# Coordenadas centrais do RJ: -22.9068, -43.1729
try:
    df_weather = fetcher.fetch_weather_sync(-22.9068, -43.1729, '2023-01-01', '2024-04-01')
    df_weather['data'] = pd.to_datetime(df_weather['data']).dt.date
    print("Clima importado com sucesso!")
except Exception as e:
    print(f"Fallback para dados mock de clima: {e}")
    # Fallback caso a API bloqueie
    dates = pd.date_range(start='2023-01-01', end='2024-04-01')
    df_weather = pd.DataFrame({
        'data': dates.date,
        'temperatura_media': np.random.normal(25, 3, len(dates)),
        'precipitacao_total': np.random.exponential(5, len(dates)) * np.random.choice([0, 1], len(dates), p=[0.7, 0.3])
    })

# Busca feriados 2023 e 2024
df_holidays_2023 = fetcher.fetch_public_holidays_sync(2023)
df_holidays_2024 = fetcher.fetch_public_holidays_sync(2024)
df_holidays = pd.concat([df_holidays_2023, df_holidays_2024], ignore_index=True)
df_holidays['data'] = df_holidays['date'].dt.date
df_holidays['eh_feriado'] = 1

# Integrando com a base
df_1746['data'] = pd.to_datetime(df_1746['data_inicio']).dt.date
df_merged = df_1746.merge(df_weather, on='data', how='left')
df_merged = df_merged.merge(df_holidays[['data', 'eh_feriado']], on='data', how='left')
df_merged['eh_feriado'] = df_merged['eh_feriado'].fillna(0)
print(f"Total de chamados em feriados: {df_merged['eh_feriado'].sum()}")

## 3. Correlação Climática

In [ ]:
# Agregar volume de chamados por dia e tipo
daily_vol = df_merged.groupby(['data', 'tipo']).size().reset_index(name='volume')
daily_weather = df_merged[['data', 'precipitacao_total']].drop_duplicates()
daily_agg = daily_vol.merge(daily_weather, on='data', how='left')

plt.figure(figsize=(10, 6))
sns.scatterplot(data=daily_agg[daily_agg['tipo'].isin(['Alagamento', 'Poda de Árvore'])], 
                x='precipitacao_total', y='volume', hue='tipo')
plt.title('Correlação: Precipitação vs Volume de Chamados (Alagamento/Poda)')
plt.xlabel('Precipitação Total (mm)')
plt.ylabel('Volume Diário')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Análise Geoespacial (Heatmap de Demanda)

In [ ]:
# Filtrar chamados válidos e recentes para visualização
df_geo = df_1746.dropna(subset=['latitude', 'longitude']).head(5000)

m = folium.Map(location=[-22.9068, -43.1729], zoom_start=11, tiles='CartoDB Positron')
heat_data = [[row['latitude'], row['longitude']] for index, row in df_geo.iterrows()]
HeatMap(heat_data, radius=10, blur=15, max_zoom=1).add_to(m)
m

## 5. Regressão para Previsão de Demanda
Modelando o volume diário com LightGBM e TimeSeriesSplit

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from src.model_utils import plot_shap_summary

daily_total = df_merged.groupby('data').agg(
    volume=('id_chamado', 'count'),
    precipitacao=('precipitacao_total', 'mean'),
    temperatura=('temperatura_media', 'mean'),
    feriado=('eh_feriado', 'max')
).reset_index()

daily_total['data'] = pd.to_datetime(daily_total['data'])
daily_total['dia_semana'] = daily_total['data'].dt.dayofweek
daily_total['mes'] = daily_total['data'].dt.month
daily_total = daily_total.sort_values('data')

X_reg = daily_total[['precipitacao', 'temperatura', 'dia_semana', 'mes', 'feriado']]
y_reg = daily_total['volume']

tscv = TimeSeriesSplit(n_splits=3)
maes = []
for train_idx, test_idx in tscv.split(X_reg):
    X_tr, X_te = X_reg.iloc[train_idx], X_reg.iloc[test_idx]
    y_tr, y_te = y_reg.iloc[train_idx], y_reg.iloc[test_idx]
    
    model_reg = LGBMRegressor(random_state=42, n_estimators=50)
    model_reg.fit(X_tr, y_tr)
    preds = model_reg.predict(X_te)
    maes.append(mean_absolute_error(y_te, preds))

print(f"MAE Médio: {np.mean(maes):.2f}")

# Treinando em tudo para o SHAP
model_reg.fit(X_reg, y_reg)
plot_shap_summary(model_reg, X_reg, feature_names=X_reg.columns.tolist())